# Adaptive Retrieval with LangGraph

## 1. Project Overview

**Task:** Build a RAG system that **decides at runtime** whether it has enough context to answer — and if not, retrieves more before responding. We use LangGraph to model this as a state machine with explicit decision nodes.

**Why this matters:** Standard RAG always retrieves the same number of documents regardless of query difficulty. Easy questions waste tokens on unnecessary context, while hard questions may not get enough. An adaptive system retrieves only what it needs.

**Stack:**
- `LangGraph` — state machine framework for agentic workflows with decision nodes
- `ChatOllama` + `qwen3.5:9b` — LLM for reasoning, judging sufficiency, and answering
- `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`) — embedding model
- `ChromaDB` — vector store for retrieval

> **No API keys required.** Everything runs locally.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | Understand **LangGraph** — state, nodes, edges, and conditional routing |
| 2 | Build a **sufficiency checker** that decides if retrieved context is enough |
| 3 | Implement **adaptive retrieval** — retrieve more only when needed |
| 4 | Design **decision nodes** that control workflow branching |
| 5 | Compare adaptive vs fixed retrieval on cost, quality, and latency |
| 6 | Handle **multi-hop questions** that need information from multiple passages |

## 3. Problem Statement

Standard RAG has a fundamental inefficiency:

```
Simple Q: "What is Docker?"     → Retrieves 5 docs (only needs 1) → wastes tokens
Hard Q:   "Compare Docker and   → Retrieves 5 docs (needs info   → misses key context
           K8s networking"         from 2 different topics)
```

**Adaptive retrieval** solves this with a feedback loop:

1. Retrieve initial context
2. Ask the LLM: *"Is this enough to answer?"*
3. If YES → generate answer
4. If NO → retrieve more (with a refined query) → go to step 2

## 4. What is LangGraph?

LangGraph models workflows as **state machines** — directed graphs where:

| Concept | What It Is | Example |
|---------|-----------|--------|
| **State** | Data flowing through the graph | `{query, documents, answer, iteration}` |
| **Node** | A function that transforms state | `retrieve()`, `check_sufficiency()`, `generate()` |
| **Edge** | Connection between nodes | `retrieve → check_sufficiency` |
| **Conditional Edge** | Edge that routes based on state | If sufficient → generate, else → retrieve_more |
| **START / END** | Entry and exit points | Graph begins at START, exits at END |

```
         ┌─────────────────────────────────────────────┐
         │              LangGraph State Machine          │
         │                                               │
  START ─┤─→ [Retrieve] ─→ [Check Sufficiency] ──YES──→ [Generate] ─→ END
         │                      │                        │
         │                      NO                       │
         │                      │                        │
         │                      ▼                        │
         │               [Refine Query] ────────────┐    │
         │                      │                   │    │
         │                      ▼                   │    │
         │               [Retrieve More] ───────────┘    │
         │                      │                        │
         │                      ▼                        │
         │               [Check Sufficiency] ──YES────→──┘
         │                      │
         │                      NO + max_iterations
         │                      ▼
         │               [Generate Best-Effort] ─→ END
         └───────────────────────────────────────────────┘
```

## 5. Environment Setup

In [ ]:
# Uncomment if any packages are missing
# !pip install -q langchain langchain-ollama langchain-core langchain-community \
#     langchain-huggingface chromadb sentence-transformers langgraph

print("Dependencies: langchain, chromadb, sentence-transformers, langgraph")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 6. Imports

## 7. Configuration

In [ ]:
import os
import re
import json
import textwrap
import warnings
from collections import Counter, defaultdict
from typing import Annotated, TypedDict

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
import chromadb

from langgraph.graph import StateGraph, START, END

LLM_MODEL = "qwen3.5:9b"
EMBED_MODEL = "all-MiniLM-L6-v2"
INITIAL_K = 2       # Start with few docs
EXPAND_K = 3        # Retrieve this many more if needed
MAX_ITERATIONS = 3  # Safety cap on retrieval loops
TEMP_JUDGE = 0.0    # Deterministic sufficiency checks
TEMP_ANSWER = 0.1   # Slight creativity for answers

print("All imports OK")
print(f"LLM: {LLM_MODEL}")
print(f"Embeddings: {EMBED_MODEL}")
print(f"Initial retrieval: {INITIAL_K} docs")
print(f"Expansion: +{EXPAND_K} docs per iteration")
print(f"Max iterations: {MAX_ITERATIONS}")

## 8. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.1) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 96):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM check
test = ask("Say 'ready'.")
print(f"LLM ready: {test[:30]}")

---

# Part A — Knowledge Base

## 9. Build the Document Corpus

30 passages covering software engineering topics. Some questions require information from **multiple passages** (multi-hop), making adaptive retrieval valuable.

In [ ]:
PASSAGES = [
    {"id": "P01", "topic": "docker",
     "text": "Docker containers package applications with their dependencies into isolated units. Unlike virtual machines, containers share the host OS kernel, making them lightweight and fast to start. Docker images are built from Dockerfiles using layered file systems where each instruction creates a new layer."},
    {"id": "P02", "topic": "docker",
     "text": "Docker Compose defines multi-container applications in a YAML file. Services, networks, and volumes are declared and managed together. The 'docker compose up' command starts all services. Environment variables, port mappings, and dependencies between services are specified declaratively."},
    {"id": "P03", "topic": "kubernetes",
     "text": "Kubernetes orchestrates containerized applications across clusters. Pods are the smallest deployable units containing one or more containers. Deployments manage desired state — specify replicas and K8s ensures they are running. Services provide stable network endpoints for groups of pods."},
    {"id": "P04", "topic": "kubernetes",
     "text": "Kubernetes networking assigns each pod a unique IP address. Services use label selectors to route traffic to matching pods. ClusterIP provides internal access, NodePort exposes on each node's IP, and LoadBalancer provisions an external load balancer. Ingress controllers manage HTTP routing rules."},
    {"id": "P05", "topic": "databases",
     "text": "PostgreSQL is a relational database supporting ACID transactions. It uses MVCC (Multi-Version Concurrency Control) for concurrent access without read locks. Write-Ahead Logging (WAL) ensures durability by writing changes to a log before applying them to data files. Indexes use B-tree structures by default."},
    {"id": "P06", "topic": "databases",
     "text": "Database connection pooling reuses existing connections instead of creating new ones for each request. PgBouncer sits between the application and PostgreSQL, maintaining a pool of connections. Session pooling assigns a connection for the duration of a client session. Transaction pooling returns connections after each transaction."},
    {"id": "P07", "topic": "databases",
     "text": "Database sharding distributes data across multiple instances for horizontal scalability. Hash-based sharding distributes rows by hashing the shard key. Range-based sharding partitions by value ranges. Cross-shard queries require scatter-gather operations. Resharding when adding nodes requires careful data migration."},
    {"id": "P08", "topic": "caching",
     "text": "Redis is an in-memory data structure store achieving sub-millisecond latency. It supports strings, lists, sets, sorted sets, and hashes. Persistence options include RDB snapshots and AOF (Append-Only File) logging. Redis Sentinel provides high availability with automatic failover."},
    {"id": "P09", "topic": "caching",
     "text": "Cache invalidation strategies include TTL (Time-To-Live) expiration, write-through (update cache and database together), and write-behind (batch writes to database asynchronously). Cache-aside loads data into cache only on miss. Cache stampede occurs when many requests simultaneously hit an expired key."},
    {"id": "P10", "topic": "api",
     "text": "REST APIs use HTTP methods for CRUD operations: GET reads, POST creates, PUT replaces, PATCH partially updates, DELETE removes. URIs identify resources as nouns. HTTP status codes indicate results: 2xx success, 4xx client error, 5xx server error. Pagination uses query parameters like limit and offset."},
    {"id": "P11", "topic": "api",
     "text": "GraphQL provides a single endpoint where clients specify exactly what data they need via typed queries. Schemas define types and relationships. Resolvers fetch data for each field. This eliminates over-fetching and under-fetching common in REST. The N+1 problem is addressed using DataLoader batching."},
    {"id": "P12", "topic": "security",
     "text": "OAuth 2.0 enables authorization without sharing credentials. The authorization code flow redirects users to an auth server, returns a code, and exchanges it for tokens. Access tokens grant API access. Refresh tokens obtain new access tokens. JWT (JSON Web Tokens) carry self-contained claims."},
    {"id": "P13", "topic": "security",
     "text": "TLS (Transport Layer Security) encrypts data in transit between client and server. The handshake establishes a shared secret using asymmetric cryptography, then switches to symmetric encryption for data transfer. Certificates verify server identity. Mutual TLS (mTLS) also authenticates the client."},
    {"id": "P14", "topic": "security",
     "text": "The OWASP Top 10 identifies critical web application security risks: injection attacks (SQL, command), broken authentication, sensitive data exposure, XML external entities, broken access control, security misconfiguration, cross-site scripting (XSS), insecure deserialization, known vulnerabilities, and insufficient logging."},
    {"id": "P15", "topic": "messaging",
     "text": "Apache Kafka is a distributed event streaming platform. Producers publish messages to topics partitioned across brokers. Consumer groups parallelize consumption — each partition is read by one consumer in the group. Kafka retains messages for a configurable period enabling replay. Kafka Connect integrates external systems."},
    {"id": "P16", "topic": "messaging",
     "text": "Message queue patterns: point-to-point delivers to exactly one consumer (task distribution). Publish-subscribe delivers to all subscribers (event broadcasting). Dead letter queues capture messages that fail processing after retries. Backpressure mechanisms prevent producers from overwhelming slow consumers."},
    {"id": "P17", "topic": "ci_cd",
     "text": "CI/CD automates the software release process. Continuous Integration runs automated builds and tests on every commit. Continuous Delivery automates deployment to staging environments. Pipeline stages include linting, unit tests, integration tests, security scanning, building artifacts, and deploying."},
    {"id": "P18", "topic": "ci_cd",
     "text": "Blue-green deployments maintain two identical production environments. Traffic is routed to the 'blue' environment while 'green' receives the new version. After verification, traffic switches to green. Canary deployments route a small percentage of traffic to the new version. Rolling updates replace instances gradually."},
    {"id": "P19", "topic": "monitoring",
     "text": "The three pillars of observability: Metrics (numerical measurements — counters, gauges, histograms), Logs (timestamped event records), and Traces (request paths across services). Prometheus collects metrics, ELK stack aggregates logs, Jaeger visualizes distributed traces."},
    {"id": "P20", "topic": "monitoring",
     "text": "SLIs (Service Level Indicators) measure service quality aspects. SLOs (Service Level Objectives) set targets for SLIs, like 99.9% availability. SLAs (Service Level Agreements) are customer contracts. Error budgets represent the allowed unreliability. Budget exhaustion signals a need to prioritize reliability."},
    {"id": "P21", "topic": "ml_ops",
     "text": "ML model serving uses REST or gRPC endpoints. TensorFlow Serving and TorchServe are dedicated model servers. Feature stores provide consistent feature computation. Model registries (MLflow) track versions. A/B testing compares model performance on live traffic."},
    {"id": "P22", "topic": "ml_ops",
     "text": "Data drift occurs when production data diverges from training data. Covariate shift changes input distributions. Concept drift changes input-output relationships. Detection methods include PSI (Population Stability Index) and KS test. Automated retraining pipelines trigger on drift detection."},
    {"id": "P23", "topic": "architecture",
     "text": "Microservices decompose applications into small, independently deployable services. Each owns its data and deployment pipeline. Benefits include independent scaling and fault isolation. Challenges include distributed transactions, network latency, and operational complexity. The strangler fig pattern enables gradual migration."},
    {"id": "P24", "topic": "architecture",
     "text": "Event sourcing stores every state change as an immutable event rather than overwriting current state. This provides a complete audit trail and enables temporal queries. CQRS separates read and write models — commands produce events, queries read optimized projections."},
    {"id": "P25", "topic": "architecture",
     "text": "The CAP theorem: distributed systems cannot simultaneously guarantee Consistency, Availability, and Partition tolerance. During network partitions, systems must choose between consistency (reject conflicting writes) and availability (accept writes that may conflict). CP systems: HBase. AP systems: Cassandra."},
    {"id": "P26", "topic": "testing",
     "text": "The testing pyramid: many fast unit tests at the base, fewer integration tests in the middle, and minimal end-to-end tests at the top. Unit tests isolate functions with mocks. Integration tests verify component interactions. E2E tests exercise complete user workflows through the full stack."},
    {"id": "P27", "topic": "testing",
     "text": "Contract testing verifies that services communicate correctly by checking message formats against agreed contracts. Pact is a popular contract testing tool. Consumer-driven contracts let API consumers define expected interactions. The provider then verifies it satisfies all consumer contracts."},
    {"id": "P28", "topic": "networking",
     "text": "Load balancers distribute traffic across servers. Layer 4 routes by IP/port. Layer 7 inspects HTTP headers for intelligent routing. Algorithms include round-robin, least connections, and weighted distribution. Health checks remove unhealthy backends. Session affinity routes a client to the same server."},
    {"id": "P29", "topic": "networking",
     "text": "DNS (Domain Name System) translates domain names to IP addresses. Recursive resolvers query root servers, TLD servers, and authoritative servers. DNS records include A (IPv4), AAAA (IPv6), CNAME (alias), MX (mail), and TXT (verification). TTL controls how long records are cached."},
    {"id": "P30", "topic": "networking",
     "text": "CDNs (Content Delivery Networks) cache content at edge locations close to users, reducing latency. Static assets (images, CSS, JS) are cached aggressively. Dynamic content is proxied to the origin server. Cache-Control and ETag headers govern freshness. Purge APIs invalidate cached content on updates."},
]

print(f"Total passages: {len(PASSAGES)}")
print(f"Topics: {dict(Counter(p['topic'] for p in PASSAGES))}")

## 10. Index Passages in ChromaDB

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("kb")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="kb",
    metadata={"hnsw:space": "cosine"},
)

texts = [p["text"] for p in PASSAGES]
metas = [{"passage_id": p["id"], "topic": p["topic"]} for p in PASSAGES]
ids = [p["id"] for p in PASSAGES]
vecs = embeddings.embed_documents(texts)

collection.add(documents=texts, embeddings=vecs, metadatas=metas, ids=ids)
print(f"Indexed {collection.count()} passages")
print(f"Embedding dim: {len(vecs[0])}")

In [ ]:
def retrieve(query: str, top_k: int = 5, exclude_ids: list[str] | None = None) -> list[dict]:
    """Retrieve passages, optionally excluding already-seen IDs."""
    # Retrieve more than needed if we're excluding some
    fetch_k = top_k + (len(exclude_ids) if exclude_ids else 0)
    qv = embeddings.embed_query(query)
    results = collection.query(query_embeddings=[qv], n_results=min(fetch_k, 30))
    hits = []
    for i in range(len(results["documents"][0])):
        pid = results["metadatas"][0][i]["passage_id"]
        if exclude_ids and pid in exclude_ids:
            continue
        hits.append({
            "passage_id": pid,
            "topic": results["metadatas"][0][i]["topic"],
            "text": results["documents"][0][i],
            "score": 1 - results["distances"][0][i],
        })
        if len(hits) >= top_k:
            break
    return hits


# Quick test
test = retrieve("Docker containers", top_k=2)
for h in test:
    print(f"  {h['passage_id']} [{h['topic']}] sim={h['score']:.3f}")

---

# Part B — LangGraph Adaptive Retrieval

## 11. Define the Graph State

The **state** carries all data through the graph. Each node reads from and writes to this shared state.

```
GraphState = {
    "question":        the user's question
    "documents":       list of retrieved passages (grows across iterations)
    "document_ids":    set of already-retrieved IDs (for dedup)
    "is_sufficient":   does the LLM think we have enough context?
    "refined_query":   if not sufficient, what to search for next
    "answer":          the final answer
    "iteration":       current retrieval iteration count
    "trace":           log of decisions for debugging
}
```

In [ ]:
class GraphState(TypedDict):
    question: str
    documents: list[dict]
    document_ids: list[str]
    is_sufficient: bool
    refined_query: str
    answer: str
    iteration: int
    trace: list[str]


print("GraphState defined with fields:")
for field in GraphState.__annotations__:
    print(f"  - {field}: {GraphState.__annotations__[field]}")

## 12. Node 1 — Initial Retrieve

The first node retrieves a **small** initial set of documents. Starting small is the key insight — we only expand if needed.

In [ ]:
def node_initial_retrieve(state: GraphState) -> dict:
    """Retrieve initial set of documents (small k)."""
    question = state["question"]
    hits = retrieve(question, top_k=INITIAL_K)
    doc_ids = [h["passage_id"] for h in hits]

    return {
        "documents": hits,
        "document_ids": doc_ids,
        "iteration": 1,
        "trace": [f"[iter 1] Initial retrieve: {doc_ids} ({INITIAL_K} docs)"],
    }


print("Node: initial_retrieve")
print(f"  Retrieves {INITIAL_K} documents on the first pass")

## 13. Node 2 — Check Sufficiency (Decision Node)

This is the **decision node** — it asks the LLM whether the retrieved documents contain enough information to answer the question. This is the core of adaptive retrieval.

The LLM acts as a **judge**, not an answerer. It only decides YES/NO and, if NO, explains what's missing.

In [ ]:
SUFFICIENCY_SYSTEM = """You decide whether retrieved documents contain enough information
to fully answer a question. You do NOT answer the question — you only judge sufficiency."""

SUFFICIENCY_PROMPT = """QUESTION: {question}

RETRIEVED DOCUMENTS:
{context}

Do these documents contain ENOUGH information to fully answer the question?
Consider: Are all parts of the question covered? Is key information missing?

Return ONLY JSON:
{{"sufficient": true/false,
  "reasoning": "explain what is covered and what is missing",
  "missing_topic": "what to search for next (empty if sufficient)"}}"""


def node_check_sufficiency(state: GraphState) -> dict:
    """Ask LLM if retrieved context is sufficient to answer."""
    question = state["question"]
    docs = state["documents"]
    iteration = state["iteration"]
    trace = list(state["trace"])

    context = "\n\n".join(
        f"[{d['passage_id']}] {d['text']}" for d in docs
    )

    resp = ask(
        SUFFICIENCY_PROMPT.format(question=question, context=context),
        system=SUFFICIENCY_SYSTEM,
        temperature=TEMP_JUDGE,
    )
    result = parse_json(resp)

    if result and "sufficient" in result:
        is_sufficient = result["sufficient"]
        reasoning = result.get("reasoning", "")
        missing = result.get("missing_topic", "")
    else:
        # Fallback: assume sufficient if we can't parse
        is_sufficient = True
        reasoning = "Could not parse judge response"
        missing = ""

    # Force sufficient if at max iterations
    if iteration >= MAX_ITERATIONS and not is_sufficient:
        trace.append(f"[iter {iteration}] Max iterations reached — forcing answer")
        is_sufficient = True

    status = "SUFFICIENT" if is_sufficient else f"INSUFFICIENT — missing: {missing[:60]}"
    trace.append(f"[iter {iteration}] Sufficiency check: {status}")

    return {
        "is_sufficient": is_sufficient,
        "refined_query": missing if not is_sufficient else "",
        "trace": trace,
    }


print("Node: check_sufficiency")
print("  LLM judges if documents are enough → routes to 'generate' or 'retrieve_more'")

## 14. Node 3 — Retrieve More

When the sufficiency check says NO, this node retrieves **additional** documents using a **refined query** that targets the missing information. Already-seen documents are excluded.

In [ ]:
def node_retrieve_more(state: GraphState) -> dict:
    """Retrieve additional documents using the refined query."""
    refined_query = state["refined_query"]
    existing_docs = list(state["documents"])
    existing_ids = list(state["document_ids"])
    iteration = state["iteration"]
    trace = list(state["trace"])

    # Use refined query if available, otherwise fall back to original
    search_query = refined_query if refined_query else state["question"]

    new_hits = retrieve(search_query, top_k=EXPAND_K, exclude_ids=existing_ids)
    new_ids = [h["passage_id"] for h in new_hits]

    # Merge with existing documents
    all_docs = existing_docs + new_hits
    all_ids = existing_ids + new_ids

    trace.append(
        f"[iter {iteration + 1}] Retrieved {len(new_ids)} more: {new_ids} "
        f"(query: '{search_query[:50]}') — total now: {len(all_docs)}"
    )

    return {
        "documents": all_docs,
        "document_ids": all_ids,
        "iteration": iteration + 1,
        "trace": trace,
    }


print("Node: retrieve_more")
print(f"  Retrieves +{EXPAND_K} additional docs, excluding already-seen")

## 15. Node 4 — Generate Answer

The final node generates an answer using **all** accumulated context.

In [ ]:
ANSWER_SYSTEM = """You answer technical questions using ONLY the provided passages.
Cite passages as [P01], [P02], etc. Be accurate, concise, and cite your sources."""


def node_generate(state: GraphState) -> dict:
    """Generate the final answer using accumulated context."""
    question = state["question"]
    docs = state["documents"]
    trace = list(state["trace"])

    context = "\n\n".join(
        f"[{d['passage_id']}] {d['text']}" for d in docs
    )

    prompt = (
        f"QUESTION: {question}\n\n"
        f"PASSAGES:\n{context}\n\n"
        "Provide a comprehensive answer citing [PXX] sources."
    )
    answer = ask(prompt, system=ANSWER_SYSTEM, temperature=TEMP_ANSWER)

    trace.append(f"[done] Generated answer using {len(docs)} documents")

    return {
        "answer": answer,
        "trace": trace,
    }


print("Node: generate")
print("  Produces final answer from all accumulated documents")

## 16. Conditional Edge — The Router

The **conditional edge** is what makes this adaptive. After the sufficiency check, it routes to either:
- `generate` (if sufficient) → END
- `retrieve_more` (if not) → loops back to `check_sufficiency`

In [ ]:
def route_after_sufficiency(state: GraphState) -> str:
    """Conditional edge: route based on sufficiency check."""
    if state["is_sufficient"]:
        return "generate"       # Enough context → answer now
    else:
        return "retrieve_more"  # Need more → fetch additional docs


print("Router: route_after_sufficiency")
print("  is_sufficient == True  → 'generate'")
print("  is_sufficient == False → 'retrieve_more'")

## 17. Assemble the Graph

Now we connect all nodes and edges into a LangGraph `StateGraph`.

```
START → initial_retrieve → check_sufficiency
                                │
                         ┌──────┴──────┐
                   sufficient?     not sufficient?
                         │              │
                    generate       retrieve_more
                         │              │
                        END        check_sufficiency (loop)
```

In [ ]:
# Build the graph
workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("initial_retrieve", node_initial_retrieve)
workflow.add_node("check_sufficiency", node_check_sufficiency)
workflow.add_node("retrieve_more", node_retrieve_more)
workflow.add_node("generate", node_generate)

# Add edges
workflow.add_edge(START, "initial_retrieve")
workflow.add_edge("initial_retrieve", "check_sufficiency")

# Conditional edge: the decision point
workflow.add_conditional_edges(
    "check_sufficiency",
    route_after_sufficiency,
    {"generate": "generate", "retrieve_more": "retrieve_more"},
)

# Loop: retrieve_more → check again
workflow.add_edge("retrieve_more", "check_sufficiency")

# End after generating
workflow.add_edge("generate", END)

# Compile
app = workflow.compile()
print("LangGraph compiled successfully")
print("Nodes: initial_retrieve → check_sufficiency ⇄ retrieve_more → generate")

## 18. Run the Adaptive Graph — Simple Question

A simple question should be answered with the **initial retrieval** alone — no extra iterations.

In [ ]:
def run_adaptive(question: str) -> dict:
    """Run the adaptive retrieval graph."""
    initial_state = {
        "question": question,
        "documents": [],
        "document_ids": [],
        "is_sufficient": False,
        "refined_query": "",
        "answer": "",
        "iteration": 0,
        "trace": [],
    }
    result = app.invoke(initial_state)
    return result


simple_q = "What is Docker and how are images built?"
print(f"Q: {simple_q}")
print("=" * 90)

result = run_adaptive(simple_q)

print(f"\nDECISION TRACE:")
for step in result["trace"]:
    print(f"  {step}")

print(f"\nFinal docs used: {result['document_ids']}")
print(f"Iterations: {result['iteration']}")
print(f"\nANSWER:")
wrap_print(result["answer"])

## 19. Run the Adaptive Graph — Multi-Hop Question

A multi-hop question requires information from **multiple topics** — the system should detect insufficiency and retrieve more.

In [ ]:
multi_hop_q = "Compare Docker container networking with Kubernetes networking — how do they differ in service discovery and load balancing?"
print(f"Q: {multi_hop_q}")
print("=" * 90)

result = run_adaptive(multi_hop_q)

print(f"\nDECISION TRACE:")
for step in result["trace"]:
    print(f"  {step}")

print(f"\nFinal docs used: {result['document_ids']}")
print(f"Iterations: {result['iteration']}")
print(f"\nANSWER:")
wrap_print(result["answer"])

## 20. Run the Adaptive Graph — Cross-Domain Question

In [ ]:
cross_domain_q = "How would you set up a complete production deployment with CI/CD pipelines, monitoring, and deployment strategies like blue-green or canary?"
print(f"Q: {cross_domain_q}")
print("=" * 90)

result = run_adaptive(cross_domain_q)

print(f"\nDECISION TRACE:")
for step in result["trace"]:
    print(f"  {step}")

print(f"\nFinal docs used: {result['document_ids']}")
print(f"Iterations: {result['iteration']}")
print(f"\nANSWER:")
wrap_print(result["answer"])

---

# Part C — Comparison: Adaptive vs Fixed

## 21. Build the Fixed Retrieval Baseline

The baseline always retrieves a fixed number of documents (5), regardless of question complexity.

In [ ]:
def run_fixed(question: str, fixed_k: int = 5) -> dict:
    """Standard fixed retrieval: always fetch fixed_k docs."""
    hits = retrieve(question, top_k=fixed_k)
    context = "\n\n".join(f"[{h['passage_id']}] {h['text']}" for h in hits)

    prompt = (
        f"QUESTION: {question}\n\n"
        f"PASSAGES:\n{context}\n\n"
        "Provide a comprehensive answer citing [PXX] sources."
    )
    answer = ask(prompt, system=ANSWER_SYSTEM, temperature=TEMP_ANSWER)

    return {
        "question": question,
        "documents": hits,
        "document_ids": [h["passage_id"] for h in hits],
        "answer": answer,
        "iteration": 1,
    }


print(f"Fixed baseline: always retrieves {5} documents")

## 22. Evaluation Set

In [ ]:
EVAL_SET = [
    # Simple (single passage needed)
    {"query": "What is Docker and how do containers work?",
     "expected_ids": ["P01"], "complexity": "simple"},
    {"query": "How does Redis achieve sub-millisecond latency?",
     "expected_ids": ["P08"], "complexity": "simple"},
    {"query": "What are the OWASP Top 10 security risks?",
     "expected_ids": ["P14"], "complexity": "simple"},
    {"query": "What is the CAP theorem?",
     "expected_ids": ["P25"], "complexity": "simple"},
    {"query": "How does DNS resolution work?",
     "expected_ids": ["P29"], "complexity": "simple"},

    # Medium (2 passages needed)
    {"query": "How do Docker and Docker Compose work together for multi-container apps?",
     "expected_ids": ["P01", "P02"], "complexity": "medium"},
    {"query": "What are the different cache invalidation strategies and how does Redis handle persistence?",
     "expected_ids": ["P08", "P09"], "complexity": "medium"},
    {"query": "Compare REST and GraphQL API design approaches.",
     "expected_ids": ["P10", "P11"], "complexity": "medium"},
    {"query": "How does CI/CD work with different deployment strategies?",
     "expected_ids": ["P17", "P18"], "complexity": "medium"},
    {"query": "How do you detect data drift and deploy ML models in production?",
     "expected_ids": ["P21", "P22"], "complexity": "medium"},

    # Complex (3+ passages needed, cross-domain)
    {"query": "Design a secure API system with authentication, encryption, and protection against common vulnerabilities.",
     "expected_ids": ["P12", "P13", "P14"], "complexity": "complex"},
    {"query": "How would you set up monitoring and reliability targets for a microservices architecture?",
     "expected_ids": ["P19", "P20", "P23"], "complexity": "complex"},
    {"query": "Explain how containers are orchestrated in Kubernetes with networking, load balancing, and service discovery.",
     "expected_ids": ["P03", "P04", "P28"], "complexity": "complex"},
    {"query": "How do you build a scalable data tier with databases, caching, and connection management?",
     "expected_ids": ["P05", "P06", "P08"], "complexity": "complex"},
    {"query": "Compare event-driven architecture patterns including Kafka, message queues, and event sourcing.",
     "expected_ids": ["P15", "P16", "P24"], "complexity": "complex"},
]

print(f"Evaluation set: {len(EVAL_SET)} queries")
print(f"Complexity: {dict(Counter(e['complexity'] for e in EVAL_SET))}")

## 23. Run Full Evaluation

In [ ]:
def compute_recall(retrieved_ids: list[str], expected_ids: list[str]) -> float:
    """What fraction of expected passages were retrieved?"""
    if not expected_ids:
        return 1.0
    found = sum(1 for eid in expected_ids if eid in retrieved_ids)
    return found / len(expected_ids)


print("Running evaluation (adaptive takes longer due to LLM sufficiency checks)...\n")

results_adaptive = []
results_fixed = []

for i, item in enumerate(EVAL_SET, 1):
    q = item["query"]

    # Adaptive
    ar = run_adaptive(q)
    a_recall = compute_recall(ar["document_ids"], item["expected_ids"])
    results_adaptive.append({
        "query": q,
        "complexity": item["complexity"],
        "expected": item["expected_ids"],
        "retrieved": ar["document_ids"],
        "recall": a_recall,
        "n_docs": len(ar["documents"]),
        "iterations": ar["iteration"],
    })

    # Fixed
    fr = run_fixed(q)
    f_recall = compute_recall(fr["document_ids"], item["expected_ids"])
    results_fixed.append({
        "query": q,
        "complexity": item["complexity"],
        "expected": item["expected_ids"],
        "retrieved": fr["document_ids"],
        "recall": f_recall,
        "n_docs": len(fr["documents"]),
        "iterations": 1,
    })

    print(f"  [{i:>2}/{len(EVAL_SET)}] [{item['complexity']:>7s}] "
          f"Adaptive: R={a_recall:.1%} ({ar['iteration']} iters, {len(ar['documents'])} docs) | "
          f"Fixed: R={f_recall:.1%} (5 docs)")

## 24. Overall Metrics

In [ ]:
avg_recall_a = sum(r["recall"] for r in results_adaptive) / len(results_adaptive)
avg_recall_f = sum(r["recall"] for r in results_fixed) / len(results_fixed)
avg_docs_a = sum(r["n_docs"] for r in results_adaptive) / len(results_adaptive)
avg_docs_f = sum(r["n_docs"] for r in results_fixed) / len(results_fixed)
avg_iters_a = sum(r["iterations"] for r in results_adaptive) / len(results_adaptive)

print("OVERALL COMPARISON")
print("=" * 65)
print(f"{'Metric':<25} {'Adaptive':>15} {'Fixed (k=5)':>15}")
print("-" * 65)
print(f"{'Avg Recall':<25} {avg_recall_a:>14.1%} {avg_recall_f:>14.1%}")
print(f"{'Avg docs retrieved':<25} {avg_docs_a:>14.1f} {avg_docs_f:>14.1f}")
print(f"{'Avg iterations':<25} {avg_iters_a:>14.1f} {'1.0':>15}")
print(f"{'LLM calls (retrieve)':<25} {avg_iters_a:>14.1f} {'0':>15}")
print(f"\nAdaptive retrieves {'fewer' if avg_docs_a < avg_docs_f else 'more'} docs "
      f"on average ({avg_docs_a:.1f} vs {avg_docs_f:.1f})")

## 25. Breakdown by Query Complexity

In [ ]:
print("METRICS BY COMPLEXITY")
print("=" * 85)

for complexity in ["simple", "medium", "complex"]:
    a_items = [r for r in results_adaptive if r["complexity"] == complexity]
    f_items = [r for r in results_fixed if r["complexity"] == complexity]

    a_recall = sum(r["recall"] for r in a_items) / len(a_items)
    f_recall = sum(r["recall"] for r in f_items) / len(f_items)
    a_docs = sum(r["n_docs"] for r in a_items) / len(a_items)
    a_iters = sum(r["iterations"] for r in a_items) / len(a_items)

    print(f"\n  {complexity.upper()} queries (n={len(a_items)}, need {len(EVAL_SET[0]['expected_ids'])} passages)")
    print(f"    {'':20s} {'Adaptive':>12} {'Fixed':>12}")
    print(f"    {'Recall':20s} {a_recall:>11.1%} {f_recall:>11.1%}")
    print(f"    {'Avg docs':20s} {a_docs:>11.1f} {'5.0':>12}")
    print(f"    {'Avg iterations':20s} {a_iters:>11.1f} {'1.0':>12}")

## 26. Per-Query Detailed Results

In [ ]:
print("PER-QUERY COMPARISON")
print("=" * 120)
print(f"  {'Query':<50} {'Cmplx':>6} {'A-Rcl':>6} {'F-Rcl':>6} {'A-doc':>6} {'A-itr':>6} {'Winner':>8}")
print("-" * 120)

adaptive_wins = 0
fixed_wins = 0
ties = 0

for ar, fr in zip(results_adaptive, results_fixed):
    q_short = textwrap.shorten(ar["query"], width=48, placeholder="...")

    if ar["recall"] > fr["recall"]:
        winner = "Adapt"
        adaptive_wins += 1
    elif fr["recall"] > ar["recall"]:
        winner = "Fixed"
        fixed_wins += 1
    else:
        # Same recall: adaptive wins if fewer docs used
        if ar["n_docs"] < fr["n_docs"]:
            winner = "Adapt*"
            adaptive_wins += 1
        else:
            winner = "Tie"
            ties += 1

    print(f"  {q_short:<50} {ar['complexity']:>6} {ar['recall']:>5.0%} {fr['recall']:>5.0%} "
          f"{ar['n_docs']:>6} {ar['iterations']:>6} {winner:>8}")

print(f"\nWins — Adaptive: {adaptive_wins} | Fixed: {fixed_wins} | Tie: {ties}")
print(f"  (* = same recall but fewer docs used)")

---

# Part D — Understanding the Decision Nodes

## 27. Deep Dive: How the Sufficiency Check Works

Let's trace through the sufficiency check for different query types to understand the decision logic.

In [ ]:
trace_queries = [
    ("What is Redis?", "Simple — should be sufficient on first try"),
    ("Compare REST and GraphQL for API design.", "Medium — may need docs from two topics"),
    ("How do you secure, test, and deploy a microservice?", "Complex — spans security, testing, CI/CD"),
]

for q, desc in trace_queries:
    print(f"\n{'='*100}")
    print(f"Q: {q}")
    print(f"Type: {desc}")
    print(f"{'='*100}")

    result = run_adaptive(q)

    print(f"\n  EXECUTION TRACE:")
    for step in result["trace"]:
        print(f"    {step}")

    print(f"\n  Final state:")
    print(f"    Documents: {result['document_ids']}")
    print(f"    Iterations: {result['iteration']}")
    print(f"    Answer preview: {result['answer'][:100]}...")

## 28. Decision Node Explained

### How the LLM Decides

The sufficiency check is a **structured prompt** that forces the LLM to:
1. Read the question carefully
2. Read all retrieved passages
3. Identify what **parts** of the question are covered
4. Identify what **parts** are missing
5. Return a YES/NO decision with reasoning

```
Question: "Compare REST and GraphQL"    ← needs info on BOTH

Retrieved docs: [P10] (REST only)        ← only one covered

LLM Judge:
  sufficient: false
  reasoning: "REST is well covered by [P10] but there is no
              information about GraphQL."
  missing_topic: "GraphQL API design query language"
                                          ↑ used as next search query
```

### Why This Works Better Than "Just Retrieve More"

| Approach | Problem |
|----------|--------|
| Always retrieve 10 docs | Wastes tokens on easy questions; still might miss the right doc |
| Retrieve, then retrieve more blindly | Might retrieve more of the same topic |
| **Adaptive with sufficiency check** | Knows *what* is missing and searches for it specifically |

## 29. LLM-as-Judge: Answer Quality Comparison

In [ ]:
JUDGE_PROMPT = """Compare these two answers to the same question.

QUESTION: {question}

ANSWER A (fixed retrieval, 5 docs): {answer_a}

ANSWER B (adaptive retrieval): {answer_b}

Score each 1-5 on: accuracy, completeness, relevance.

Return ONLY JSON:
{{"answer_a": {{"accuracy": N, "completeness": N, "relevance": N}},
  "answer_b": {{"accuracy": N, "completeness": N, "relevance": N}},
  "winner": "A or B or TIE",
  "explanation": "why"}}"""

judge_qs = [
    "Compare REST and GraphQL for API design.",
    "Design a secure API system with authentication and encryption.",
    "How does Redis achieve fast performance and data persistence?",
]

print("ANSWER QUALITY COMPARISON (LLM-as-Judge)")
print("=" * 90)

for q in judge_qs:
    ar = run_adaptive(q)
    fr = run_fixed(q)

    resp = ask(
        JUDGE_PROMPT.format(
            question=q,
            answer_a=fr["answer"][:400],
            answer_b=ar["answer"][:400],
        ),
        system="You evaluate answer quality.",
        temperature=TEMP_JUDGE,
    )
    scores = parse_json(resp)
    if scores:
        print(f"\n  Q: {textwrap.shorten(q, 65, placeholder='...')}")
        print(f"  Fixed docs: {fr['document_ids'][:5]}")
        print(f"  Adaptive:   {ar['document_ids']} ({ar['iteration']} iters)")
        a = scores.get("answer_a", {})
        b = scores.get("answer_b", {})
        print(f"  Fixed scores:    acc={a.get('accuracy','?')} comp={a.get('completeness','?')} rel={a.get('relevance','?')}")
        print(f"  Adaptive scores: acc={b.get('accuracy','?')} comp={b.get('completeness','?')} rel={b.get('relevance','?')}")
        print(f"  Winner: {scores.get('winner', '?')} — {str(scores.get('explanation', ''))[:80]}")

---

# Part E — Analysis & Learning Notes

## 30. Efficiency Analysis

The key insight: adaptive retrieval uses **fewer total tokens** on simple questions and **more targeted tokens** on complex ones.

In [ ]:
print("EFFICIENCY ANALYSIS")
print("=" * 70)

for complexity in ["simple", "medium", "complex"]:
    a_items = [r for r in results_adaptive if r["complexity"] == complexity]
    f_items = [r for r in results_fixed if r["complexity"] == complexity]

    a_docs = sum(r["n_docs"] for r in a_items) / len(a_items)
    f_docs = sum(r["n_docs"] for r in f_items) / len(f_items)
    a_recall = sum(r["recall"] for r in a_items) / len(a_items)
    f_recall = sum(r["recall"] for r in f_items) / len(f_items)
    a_iters = sum(r["iterations"] for r in a_items) / len(a_items)

    savings = (f_docs - a_docs) / f_docs * 100 if f_docs else 0
    recall_delta = a_recall - f_recall

    print(f"\n  {complexity.upper()} queries (n={len(a_items)})")
    print(f"    Adaptive avg docs: {a_docs:.1f} | Fixed: {f_docs:.0f} | "
          f"{'Saved' if savings > 0 else 'Extra'}: {abs(savings):.0f}% tokens")
    print(f"    Recall: Adaptive={a_recall:.0%} Fixed={f_recall:.0%} (Δ{recall_delta:+.0%})")
    print(f"    Avg iterations: {a_iters:.1f}")

## 31. Common Mistakes

| Mistake | Why It's Wrong | Better Approach |
|---------|---------------|----------------|
| No max iteration cap | LLM might loop forever if never satisfied | Always set MAX_ITERATIONS (2-3 is typical) |
| Sufficiency check as yes/no without reasoning | Can't target what's missing for the next retrieval | Always ask the LLM what specific topic is missing |
| Re-retrieving with the same query | Gets the same documents again | Use the refined query targeting missing topics AND exclude already-seen IDs |
| Overly strict sufficiency threshold | Almost everything triggers extra retrieval | Tune the prompt to accept "good enough" context |
| Not tracking iteration history | Can't debug why the system looped or stopped early | Maintain a trace log in the state |
| Using adaptive for all queries | Simple questions pay the latency cost of sufficiency checks | Classify complexity first; use fixed for simple queries |

## 32. Production Ideas

1. **Complexity classifier** — predict query complexity upfront, skip adaptive for simple queries
2. **Confidence scoring** — use LLM confidence (not just sufficiency) to decide if more context needed
3. **Parallel retrieval** — send original + refined queries simultaneously instead of sequentially
4. **Adaptive k** — start with k=1 for likely-simple queries, k=3 for likely-complex ones
5. **Human-in-the-loop** — if the system is unsure after max iterations, ask the user to clarify
6. **Cached sufficiency patterns** — learn which query patterns typically need extra retrieval
7. **Token budget awareness** — factor remaining context window into sufficiency decisions

## 33. Exercises

### Exercise 1
Add a **query rewriting node** before retrieval: if the sufficiency check fails, rewrite the query (not just refine the search term) before the next retrieval. Compare with the current refined-query approach.

### Exercise 2
Implement a **complexity predictor** node: before any retrieval, classify the query as simple/medium/complex and set `INITIAL_K` accordingly (1/3/5). Measure if this reduces total iterations.

### Exercise 3
Add **source verification**: after generating the answer, add a node that checks whether each cited passage actually supports the claim made. If not, regenerate with a warning.

### Mini Challenge
Extend the graph with a **fallback to web search** node: if after MAX_ITERATIONS the system still lacks sufficient context, simulate a web search (use a different collection) for additional information. Add this as a new branch in the conditional routing.

In [ ]:
print("SESSION SUMMARY")
print("=" * 65)
print(f"Knowledge base: {len(PASSAGES)} passages, {len(set(p['topic'] for p in PASSAGES))} topics")
print(f"Evaluation: {len(EVAL_SET)} queries (simple/medium/complex)")
print(f"")
print(f"ADAPTIVE vs FIXED RETRIEVAL")
print(f"  {'Metric':<25} {'Adaptive':>12} {'Fixed':>12}")
print(f"  {'-'*49}")
print(f"  {'Avg Recall':<25} {avg_recall_a:>11.1%} {avg_recall_f:>11.1%}")
print(f"  {'Avg docs used':<25} {avg_docs_a:>11.1f} {avg_docs_f:>11.1f}")
print(f"  {'Avg iterations':<25} {avg_iters_a:>11.1f} {'1.0':>12}")
print(f"")
print(f"LangGraph components built:")
print(f"  Nodes:")
print(f"    initial_retrieve  — fetch small initial doc set")
print(f"    check_sufficiency — LLM decides if context is enough")
print(f"    retrieve_more     — fetch additional targeted docs")
print(f"    generate          — produce final answer")
print(f"  Edges:")
print(f"    START → initial_retrieve → check_sufficiency")
print(f"    check_sufficiency → generate (if sufficient)")
print(f"    check_sufficiency → retrieve_more (if not)")
print(f"    retrieve_more → check_sufficiency (loop)")
print(f"    generate → END")

## 34. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Fixed retrieval wastes tokens** on simple questions and misses context on complex ones |
| 2 | **Adaptive retrieval** uses a sufficiency check to retrieve only what's needed |
| 3 | **LangGraph** models this cleanly as a state machine with nodes, edges, and conditional routing |
| 4 | The **decision node** is the key component — it judges context quality and controls the loop |
| 5 | **Refined queries** (what's missing) beat re-querying with the same terms |
| 6 | **Exclude already-seen IDs** to avoid retrieving duplicate passages |
| 7 | Always set **MAX_ITERATIONS** (2-3) to prevent infinite loops |
| 8 | Adaptive retrieval shines on **multi-hop, cross-domain questions** |
| 9 | The **trace log** is essential for debugging decision flow in production |
| 10 | In production, combine with **complexity classification** to skip adaptive overhead for simple queries |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*